In [0]:
default_dataset_path = "/Volumes/loan_default_workspace/default/raw_data/Loan_Default.csv"

default_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(default_dataset_path)
)

display(default_df)


In [0]:
print(f"Rows: {default_df.count()}")
print(f"Columns: {len(default_df.columns)}")
default_df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

# Count nulls in every column
null_counts = default_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in default_df.columns
])

null_counts.show(vertical=True)

In [0]:
from pyspark.sql.functions import count, when

default_df.select(
    count("*").alias("total_loans"),
    spark_sum(col("Status")).alias("total_defaults"),
    (spark_sum(col("Status")) / count("*") * 100).alias("default_rate_%")
).show()

In [0]:
# Register the dataframe as a SQL table called "loans"
# What this does: makes your data queryable with normal SQL
default_df.createOrReplaceTempView("loans")

In [0]:
%sql
SELECT 
    Region,
    COUNT(*) AS total_loans,
  Sum(Status) AS total_defaults  -- you fill this in
FROM loans
GROUP BY Region

In [0]:
%sql
SELECT 
    Status,
    AVG(Credit_Score) AS avg_credit_score,
    COUNT(*) AS total_loans
FROM loans
GROUP BY Status

In [0]:
%sql
SELECT 
    Region,
    COUNT(*) AS total_loans,
    SUM(Status) AS total_defaults,
    ROUND(SUM(Status) / COUNT(*) * 100, 2) AS default_rate_pct
FROM loans
GROUP BY Region
ORDER BY default_rate_pct DESC

In [0]:
display(default_df)

In [0]:
%sql
SELECT 
    Status,
    ROUND(AVG(dtir1), 2) AS avg_dtir1,
    COUNT(*) AS total_loans
FROM loans
GROUP BY Status